训练基于分子描述符或者分子指纹的模型。
- 分子特征。
    - 分子指纹
    - 分子描述符
    - 分子图
    - 分子3D图
- 分子对接能量标签。
- 模型选择
    - MLP
    - GCN
    - Molecule GNN
    - EGNN

In [1]:
# 设置工作目录
import os
import gzip
import pickle
import pandas as pd

os.chdir("../data/dataset/outputs")
# 设置python工作目录
import sys

sys.path.append("/home/huabei/project/SMTarRNA")
os.listdir()

['3a6p_1m.tar.gz',
 '3a6p',
 '3a6p-exhaustiveness',
 '3a6p_1m',
 '4z4c',
 '4z4c_1m',
 '4z4c_1m.tar.gz',
 '4z4d_1m',
 '4z4d_1m.tar.gz',
 '6cbd_1m',
 '6cbd_1m.tar.gz',
 'readme.md',
 'test.csv',
 'test.txt',
 'test_out.csv',
 'zinc20_druglike_random_sample_molecule_1f600.pkl',
 'zinc_id_1m.txt',
 'zinc_id_32k_curl_smi.smi',
 'zinc_id_32k_curl_smi.smi.bk']

## 汇总对接结果

In [2]:
# data_dir = "3a6p_1m/"
# data_dir = "4z4c_1m/"
# data_dir = "4z4d_1m/"
data_dir = "6cbd_1m/"
data_files = os.listdir(data_dir)
pkl_files = [f for f in data_files if f.endswith(".pkl")]
pdbqt_gz_files = [f for f in data_files if f.endswith(".pdbqt.gz")]
len(pkl_files), len(pdbqt_gz_files), pkl_files[:5], pdbqt_gz_files[:5]

(100,
 100,
 ['zinc20_druglike_random_sample_molecule_1f600_6cbd_dock_energy_942403-952119_64_20230819142613.pkl',
  'zinc20_druglike_random_sample_molecule_1f600_6cbd_dock_energy_437197-446913_64_20230812183621.pkl',
  'zinc20_druglike_random_sample_molecule_1f600_6cbd_dock_energy_446913-456628_64_20230812183652.pkl',
  'zinc20_druglike_random_sample_molecule_1f600_6cbd_dock_energy_456628-466344_64_20230812183652.pkl',
  'zinc20_druglike_random_sample_molecule_1f600_6cbd_dock_energy_466344-476059_64_20230812183653.pkl'],
 ['zinc20_druglike_random_sample_molecule_1f600_6cbd_dock_results_87439-97155_64_20230812183320.pdbqt.gz',
  'zinc20_druglike_random_sample_molecule_1f600_6cbd_dock_results_874395-884110_64_20230812184025.pdbqt.gz',
  'zinc20_druglike_random_sample_molecule_1f600_6cbd_dock_results_884110-893826_64_20230812184025.pdbqt.gz',
  'zinc20_druglike_random_sample_molecule_1f600_6cbd_dock_results_893826-903541_64_20230812184025.pdbqt.gz',
  'zinc20_druglike_random_sample_molec

In [3]:
total_data = dict()
for f in pkl_files:
    with open(os.path.join(data_dir, f), "rb") as f:
        data = pickle.load(f)
        total_data.update(data)
print(f"total data: {len(total_data)}")

total data: 971167


In [6]:
with open(f"{data_dir[:-1]}_total_data_dock_energy.pkl", "wb") as f:
    pickle.dump(total_data, f)

In [7]:
# 提取最佳对接能量
total_data_best = {k: v[0] for k, v in total_data.items()}
# 生成最佳能量表
total_data_best_df = pd.DataFrame.from_dict(
    total_data_best,
    columns=["total", "inter", "intra", "torsions", "intra best pose"],
    orient="index",
)
total_data_best_df.head()

,total,inter,intra,torsions,intra best pose
ZINC000742895225,-9.516,-11.252,-0.316,2.225,0.174
ZINC000742915316,-8.880,-11.154,-1.046,2.596,-0.724
ZINC000772479708,-8.422,-13.288,-0.103,4.431,-0.538
ZINC000770584883,-9.703,-13.425,-1.230,3.971,-0.982
ZINC000786932805,-8.860,-10.045,1.075,2.072,1.962


## 分子特征


### 分子指纹

In [2]:
# 获取训练样本的数据
structure_file = '../ligand/zinc20_druglike_random_sample_molecule_1f600.pdbqt.gz'

In [19]:
## pdbqt to smiles
def pdbqt_to_smiles(structure_file: str):
    from openbabel.pybel import readstring
    with gzip.open(structure_file, 'rb') as f:
        pdbqt_str = f.read().decode('utf-8')
    pdbqt_mols_str = pdbqt_str.strip().split('ENDMDL\n')
    smileses = []
    zinc_ids = []
    for mol_str in pdbqt_mols_str:
        mol = readstring('pdbqt', mol_str)
        smiles, zinc_id = mol.write('smi').strip().split('\t')
        rd_mol = Chem.MolFromSmiles(smiles)
        if rd_mol is None:
            # continue
            sm = None
        else:
            sm = Chem.MolToSmiles(rd_mol)
        smileses.append(sm)
        zinc_ids.append(zinc_id)
    return pd.DataFrame({'zinc_id': zinc_ids, 'smiles': smileses}).set_index('zinc_id')

from rdkit import Chem
from rdkit.Chem import AllChem
fg_gen = AllChem.GetRDKitFPGenerator()
# smiles to fingerprint
def smiles_to_fingerprint(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return fg_gen.GetFingerprint(mol)
from skfp.fingerprints import ECFPFingerprint
fp_ecfp_transformer = ECFPFingerprint(fp_size=2048, radius=3, n_jobs=-1)
def cal_fp(smileses, batch_size=1000):
    fps = []
    for i in range(0, len(smileses), batch_size):
        batch_smiles = smileses[i:i + batch_size]
        batch_fps = fp_ecfp_transformer.transform(batch_smiles)
        fps.extend(batch_fps)
    return fps

In [ ]:
zinc_1_600_smiles = pdbqt_to_smiles(structure_file)
zinc_1_600_smiles.head()

In [ ]:
zinc_1_600_smiles.info()

In [6]:
# 保存不能够转换为smiles的zinc id 到txt
# 使用xargs和curl并行下载对应的smiles
pdbqt_cng_smi_zinc_ids = zinc_1_600_smiles[zinc_1_600_smiles['smiles'].isnull()].index.to_list()
with open('zinc_id_1m_no_smi.txt', 'w') as f:
    zinc_id_1m_n = [id + '\n' for id in pdbqt_cng_smi_zinc_ids]
    f.writelines(zinc_id_1m_n)

In [8]:
# 下载后的smiles和zinc id 文件
downloaded_smiles = []
with open('zinc_id_32k_curl_smi.smi') as f:
    for line in f:
        smiles, zinc_id = line.strip().split(' ')
        downloaded_smiles.append((smiles, zinc_id))
# 创建DataFrame并删除重复的索引标签
pdbqt_cng_smi = pd.DataFrame(downloaded_smiles, columns=['smiles', 'zinc_id']).drop_duplicates(subset=['zinc_id']).set_index('zinc_id')

# 更新原始表格为None的值
zinc_1_600_smiles.update(pdbqt_cng_smi, overwrite=False)
zinc_1_600_smiles.info()


<class 'pandas.core.frame.DataFrame'>
Index: 971561 entries, ZINC000057453708 to ZINC001461654371
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   smiles  971561 non-null  object
dtypes: object(1)
memory usage: 14.8+ MB


In [22]:
# 计算分子指纹
# zinc_1_600_smiles['fingerprint'] = zinc_1_600_smiles['smiles'].apply(smiles_to_fingerprint)
# use skfp
# cal_fp(zinc_1_600_smiles['smiles'][:1000].to_list(), 100)
zinc_1_600_smiles['fingerprint'] = cal_fp(zinc_1_600_smiles['smiles'].to_list(), 1000)

# zinc_id_1m = zinc_1_600_smiles.index.to_list()
# with open('zinc_id_1m.txt', 'w') as f:
#     zinc_id_1m_n = [id + '\n' for id in zinc_id_1m]
#     f.writelines(zinc_id_1m_n)

In [25]:
# 保存hdf数据
zinc_1_600_smiles.to_hdf('zinc_id_smiles_ecfp.h5', key='data', mode='w')

/tmp/ipykernel_1690198/1619510533.py:2: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['smiles', 'fingerprint'], dtype='object')]

  zinc_1_600_smiles.to_hdf('zinc_id_smiles_ecfp.h5', key='data', mode='w')


### 准备数据

In [ ]:
# 分子表征
zinc_1_600_smiles_ecfp = pd.read_hdf('zinc_id_smiles_ecfp.h5', key='data')

In [4]:
cmpx = '3a6p'
dock_energy = pickle.load(open(f'{cmpx}_1m_total_data_dock_energy.pkl', 'rb'))
# 提取最佳对接能量
total_data_best = {k: v[0] for k, v in dock_energy.items()}
# 生成最佳能量表
total_data_best_df = pd.DataFrame.from_dict(
    total_data_best,
    columns=["total", "inter", "intra", "torsions", "intra best pose"],
    orient="index",
)
total_data_best_df.head()

,total,inter,intra,torsions,intra best pose
ZINC000959424384,-6.857,-9.243,-1.069,2.405,-1.049
ZINC000056904143,-7.029,-8.725,-0.565,1.644,-0.617
ZINC001590300828,-8.658,-10.179,-0.055,1.012,-0.564
ZINC001416700721,-6.884,-10.205,-0.965,3.220,-1.066
ZINC000666520864,-6.403,-9.929,-0.865,3.369,-1.022


In [5]:
# 合并表
total_data_best_df.index.name = "zinc_id"
total_data_best_df = zinc_1_600_smiles_ecfp.join(total_data_best_df, how="left")
total_data_best_df.head()

,smiles,fingerprint,total,inter,intra,torsions,intra best pose
zinc_id,,,,,,,
ZINC000000006251,C=C1CCN([C@H](C)[C@](O)(Cn2cncn2)c2ccc(F)cc2F)CC1,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-7.326,-9.373,-1.070,2.355,-0.762
ZINC000000023541,O=C(N[C][C]1[C][C][C][C][C]1)Nc1[c]c(F)[c][c]c1F,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",-6.764,-7.947,-0.264,1.186,-0.260
ZINC000000027943,[C][C]OC(=O)C1=[C]N=C2[C](Br)C(=O)[N]N2[C]1[C],"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-6.242,-7.414,-0.065,1.095,-0.142
ZINC000000029829,[C][C]1[C][C]([C])[C]N(S(=O)(=O)[C]c2[c][c][c]...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-6.333,-7.251,-0.638,1.111,-0.445
ZINC000000030076,[C]C(=O)Nc1[c][c]c(C(=O)Nc2[c][c][c][c]c2[C])[...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-6.643,-7.823,-0.261,1.165,-0.276
